# 01 - Chargement et Pre-traitement

Objectif : charger les images IRM depuis `data/raw` et preparer les donnees.

- Chaque dossier = une classe (glioma, meningioma, notumor, pituitary)
- On verifie les extensions, on gere les images corrompues
- On redimensionne en 224x224 et on split train/test

Fonctions testees : `chargement_pretraitement_dataset`, `preparer_donnees`

In [1]:
import os
import sys
from pathlib import Path

# Permet d'importer src depuis notebooks/
PROJECT_ROOT = Path().resolve().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.data_preprocessing import chargement_pretraitement_dataset, preparer_donnees

print("Imports OK")

2026-10-17 00:44:58.539234: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
2026-10-17 00:44:58.606004: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-10-17 00:45:00.496849: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-10-17 00:45:05.738138: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off,

Imports OK


In [2]:
# 1. Charger les images
DATA_PATH = PROJECT_ROOT / "data" / "raw"
print(f"Dataset : {DATA_PATH} - existe ? {DATA_PATH.exists()}")
if DATA_PATH.exists():
    for cls in sorted(os.listdir(DATA_PATH)):
        p = DATA_PATH / cls
        if p.is_dir():
            print(f" - {cls} : {len(list(p.glob('*')))} fichiers")

Dataset : /app/data/raw - existe ? True
 - glioma : 1621 fichiers
 - meningioma : 1645 fichiers
 - notumor : 2000 fichiers
 - pituitary : 1757 fichiers


In [ ]:
# 2. Test de la fonction de chargement
X, y = chargement_pretraitement_dataset(str(DATA_PATH), taille_image=224)
print(f"\nX shape : {X.shape} - dtype {X.dtype}")
print(f"y shape : {y.shape} - exemples {y[:5]}")
print(f"Classes : {set(y)}")

Classes detectees : ['.gitkeep', 'glioma', 'meningioma', 'notumor', 'pituitary']
 -> Classe : glioma


In [ ]:
# 3. Preparer les donnees (encode + split + normalisation)
X_train, X_test, y_train, y_test, le, class_weights = preparer_donnees(X, y, test_size=0.2)

print(f"\nX_train : {X_train.shape} - {X_train.dtype} - min {X_train.min():.2f} max {X_train.max():.2f}")
print(f"X_test  : {X_test.shape}")
print(f"y_train : {y_train.shape} - y_test : {y_test.shape}")
print(f"LabelEncoder classes : {list(le.classes_)}")
print(f"Poids : {class_weights}")

### Resultat attendu
- X_train et X_test normalises entre 0 et 1
- y en one-hot (ex: glioma -> [1,0,0,0])
- Split 80/20 avec stratify (meme proportion par classe)